In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional, Any
import mlflow
import mlflow.pyfunc
import re
import math
from bs4 import BeautifulSoup
import pandas as pd

# --------------------------------------------------
# ✅ INIT API
# --------------------------------------------------
app = FastAPI(title="Rakuten Product Classification API")

# --------------------------------------------------
# ✅ MLflow CONFIG
# --------------------------------------------------
mlflow.set_tracking_uri("file:///C:/Users/user/Rakuten-Challenge/mlruns")

# ✅ modèle en production
model = mlflow.pyfunc.load_model("models:/rakuten_model@prod")

print("=== DEBUG LOADED MODEL ===")

try:
    # pour mlflow.sklearn: accès au modèle sklearn sous-jacent
    sk_model = model._model_impl.sklearn_model
    print("Sklearn model type:", type(sk_model))

    # si c'est un Pipeline, on peut inspecter les steps
    if hasattr(sk_model, "steps"):
        print("Pipeline steps:", [name for name, _ in sk_model.steps])

        for name, step in sk_model.steps:
            if hasattr(step, "preprocessor"):
                print(f"Step {name} has preprocessor:", type(step.preprocessor))
except Exception as e:
    print("DEBUG cannot inspect sklearn internals:", repr(e))


# --------------------------------------------------
# ✅ DATA MODEL
# --------------------------------------------------
class Product(BaseModel):
    designation: str
    description: Optional[str] = ""

# --------------------------------------------------
# ✅ CLEANING HELPERS (robuste)
# --------------------------------------------------
def safe_to_str(x: Any) -> str:
    """
    Convertit n'importe quel type en string propre
    et gère NaN/None.
    """
    if x is None:
        return ""
    # numpy/pandas NaN -> float('nan')
    if isinstance(x, float) and math.isnan(x):
        return ""
    return str(x)

def clean_text(text: str) -> str:
    # sécurités: on force bien du str
    text = safe_to_str(text)

    text = BeautifulSoup(text, "html.parser").get_text()
    text = text.lower()
    text = re.sub(r"[^a-zàâçéèêëîïôûùüÿñæœ0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --------------------------------------------------
# ✅ HEALTH CHECK
# --------------------------------------------------
@app.get("/health")
def health():
    return {"status": "ok"}

# --------------------------------------------------
# ✅ SINGLE PREDICTION
# --------------------------------------------------

@app.post("/predict")
def predict(product: Product):
    designation = safe_to_str(product.designation)
    description = safe_to_str(product.description)

    raw_text = f"{designation} {description}".strip()
    text = clean_text(raw_text).strip()

    if not text:
        raise HTTPException(status_code=400, detail="Text is empty after preprocessing")

    # ✅ IMPORTANT: le modèle attend list[str]
    X_in = [str(text)]

    try:
        pred = model.predict(X_in)[0]
        return {"prediction": int(pred)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"ML error: {e}")

# --------------------------------------------------
# ✅ BATCH PREDICTION
# --------------------------------------------------
@app.post("/predict_batch")
def predict_batch(products: List[Product]):
    if len(products) == 0:
        raise HTTPException(status_code=400, detail="Empty input list")

    texts = []
    for p in products:
        designation = safe_to_str(p.designation)
        description = safe_to_str(p.description)

        raw_text = f"{designation} {description}".strip()
        cleaned = clean_text(raw_text).strip()

        if cleaned:  # on garde seulement si non vide
            texts.append(str(cleaned))

    if len(texts) == 0:
        raise HTTPException(status_code=400, detail="All inputs are empty after preprocessing")

    try:
        preds = model.predict(texts)
        return {"predictions": [int(x) for x in preds.tolist()]}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"ML error: {e}")



=== DEBUG LOADED MODEL ===
Sklearn model type: <class 'sklearn.pipeline.Pipeline'>
Pipeline steps: ['vectorizer', 'model']
Step vectorizer has preprocessor: <class 'NoneType'>
